# 02 - Heuristic & Metaheuristic Optimization: Clarke-Wright & Simulated Annealing

**Member 2 (Kavishka) Individual Implementation & Experiments**

## 1. Problem Context & Heuristic Justification
The **Capacitated Vehicle Routing Problem (CVRP)** is NP-hard. Exact solvers (such as Mixed-Integer Linear Programming via Branch-and-Bound/Branch-and-Cut) face combinatorial explosion: on instances with over 50 customers, exact solvers can take hours or hit memory limits without finding a provably optimal solution.

To solve real-world logistics challenges at scale, **Heuristics and Metaheuristics** provide high-quality, feasible solutions in seconds. In this notebook, we implement and evaluate a 3-stage heuristic pipeline:
1. **Construction Heuristic:** Clarke-Wright Savings algorithm.
2. **Local Search:** Variable Neighborhood Descent (Intra-route 2-Opt, Inter-route Relocate, Inter-route Swap).
3. **Metaheuristic:** Simulated Annealing with geometric cooling to escape local minima.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.checker import check_solution, solution_cost, route_cost
from src.data_loader import load_instance, load_bks, instance_table
from src.clarke_wright import clarke_wright_savings
from src.local_search import two_opt_route, apply_2opt_all, run_local_search
from src.simulated_annealing import SimulatedAnnealingCVRP
from src.heuristics import solve_clarke_wright, solve_local_search, solve_simulated_annealing

FIG_DIR = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Environment initialized successfully.")

## 2. Construction Heuristic: Clarke-Wright Savings
The algorithm starts with $n$ back-and-forth routes: $[0, i, 0]$ for each customer $i$.
For each pair $(i, j)$, the savings from combining them into a single route is:
$$s_{ij} = c_{0i} + c_{0j} - \lambda c_{ij}$$
Customer pairs are sorted in descending order of savings and merged subject to:
1. Capacity constraint: $\text{load}(R_i) + \text{load}(R_j) \le Q$.
2. Exterior condition: Both $i$ and $j$ must be adjacent to the depot in their current routes.
3. Fleet size limit: Total routes $\le K$.

In [ ]:
inst_name = "A-n32-k5"
inst = load_instance(inst_name)
bks_routes, bks_cost = load_bks(inst_name)

cw_routes, cw_cost, cw_time, cw_chk = solve_clarke_wright(inst)
cw_gap = 100.0 * (cw_cost - bks_cost) / bks_cost

print(f"Instance: {inst_name} (Customers: {inst.n_customers}, Capacity: {inst.capacity}, Fleet K: {inst.max_vehicles})")
print(f"Optimal Cost (BKS): {int(bks_cost)}")
print(f"Clarke-Wright Cost: {cw_cost} (Gap: {cw_gap:.2f}%) in {cw_time:.4f}s")
print(f"Feasibility: {cw_chk.feasible} | Routes: {cw_chk.n_routes} | Route Loads: {cw_chk.loads}")

## 3. Local Search Improvement: Neighborhood Operators
Greedy construction leaves room for local improvements. We apply three neighborhood operators:
* **Intra-route 2-Opt:** Inverts a sub-segment $[v_i, \dots, v_j]$ within a route to untangle crossing paths.
* **Inter-route Relocate (1-0 Shift):** Moves a customer from one route into another where spare capacity exists.
* **Inter-route Swap (1-1 Exchange):** Swaps two customers between two different routes, balancing vehicle loads.

In [ ]:
ls_routes, ls_cost, ls_time, ls_chk = solve_local_search(inst, initial_routes=cw_routes)
ls_gap = 100.0 * (ls_cost - bks_cost) / bks_cost

print(f"Local Search Cost: {ls_cost} (Gap: {ls_gap:.2f}%) in {ls_time:.4f}s")
print(f"Improvement over Clarke-Wright: {cw_cost - ls_cost} units ({(cw_cost - ls_cost)/cw_cost * 100:.2f}% reduction)")
print(f"Feasibility: {ls_chk.feasible} | Route Loads: {ls_chk.loads}")

## 4. Metaheuristic: Simulated Annealing (SA)
Local Search terminates as soon as it reaches a local minimum. Simulated Annealing avoids premature convergence by probabilistic exploration:
* Improving moves ($\Delta \le 0$) are always accepted.
* Worsening moves ($\Delta > 0$) are accepted with Metropolis probability:
$$P(\text{accept}) = \exp\left(-\frac{\Delta}{T}\right)$$
* Temperature decays geometrically: $T_{k+1} = \alpha \cdot T_k$, where $\alpha \in [0.95, 0.99]$.

In [ ]:
sa = SimulatedAnnealingCVRP(cooling_rate=0.985, max_iterations=8000, seed=42)
sa_res = sa.solve(inst)
sa_gap = 100.0 * (sa_res.best_cost - bks_cost) / bks_cost

print(f"Simulated Annealing Best Cost: {sa_res.best_cost} (Gap: {sa_gap:.2f}%) in {sa_res.runtime_seconds:.3f}s")
print(f"Iterations: {sa_res.iterations_run} | Accepted Moves: {sa_res.history.accepted_moves}")
print(f"Feasibility: {sa_res.check_result.feasible}")

In [ ]:
h = sa_res.history
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(h.iterations, h.current_costs, label="Current Solution", color="#aec7e8", alpha=0.7)
ax1.plot(h.iterations, h.best_costs, label="Best-So-Far", color="#1f77b4", linewidth=2.5)
ax1.axhline(bks_cost, color="green", linestyle="--", linewidth=1.8, label=f"Optimal BKS ({int(bks_cost)})")
ax1.set_xlabel("Iteration", fontsize=12)
ax1.set_ylabel("Solution Cost", color="#1f77b4", fontsize=12)
ax1.tick_params(axis="y", labelcolor="#1f77b4")
ax1.grid(True, linestyle="--", alpha=0.4)

ax2 = ax1.twinx()
ax2.plot(h.iterations, h.temperatures, color="#d62728", linestyle=":", linewidth=1.5, label="Temperature T")
ax2.set_ylabel("Temperature T", color="#d62728", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#d62728")

ax1.set_title(f"Simulated Annealing Convergence Profile on {inst_name}", fontsize=14, fontweight="bold")
ax1.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 5. Parameter Tuning & Sensitivity Study
The cooling rate $\alpha$ governs the balance between **exploration** (high temperature, searching broadly) and **exploitation** (low temperature, refining locally).
We evaluate $\alpha \in [0.90, 0.95, 0.985, 0.995]$ on `A-n32-k5`.

In [ ]:
alphas = [0.90, 0.95, 0.985, 0.995]
colors = ["#d95f02", "#7570b3", "#1b9e77", "#e7298a"]
fig, ax = plt.subplots(figsize=(10, 5))

for alpha, col in zip(alphas, colors):
    runner = SimulatedAnnealingCVRP(cooling_rate=alpha, max_iterations=8000, seed=42)
    res = runner.solve(inst)
    gap = 100.0 * (res.best_cost - bks_cost) / bks_cost
    ax.plot(res.history.iterations, res.history.best_costs, label=f"alpha={alpha} (Best={res.best_cost}, Gap={gap:.1f}%)", color=col, linewidth=2)

ax.axhline(bks_cost, color="black", linestyle="--", label=f"Optimal BKS ({int(bks_cost)})")
ax.set_xlabel("Iteration", fontsize=12)
ax.set_ylabel("Best Cost", fontsize=12)
ax.set_title("Sensitivity Analysis: Cooling Rate alpha", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 6. Route Structure Visualization
Visualizing the delivery tours generated by Simulated Annealing compared against the proven optimal solution.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
cmap = plt.get_cmap("tab10")

# Optimal BKS
ax1.scatter(inst.coords[0, 0], inst.coords[0, 1], c="black", s=140, marker="s", zorder=5, label="Depot")
ax1.scatter(inst.coords[1:, 0], inst.coords[1:, 1], c="gray", s=35, zorder=4, label="Customer")
for r_i, route in enumerate(bks_routes):
    col = cmap(r_i % 10)
    path = [0, *route, 0]
    pts = inst.coords[path]
    ax1.plot(pts[:, 0], pts[:, 1], color=col, linewidth=2, alpha=0.85)
ax1.set_title(f"Optimal Solution (BKS)\nCost: {int(bks_cost)} | Routes: {len(bks_routes)}", fontsize=12, fontweight="bold")
ax1.grid(True, linestyle=":", alpha=0.4)
ax1.legend(loc="upper left")

# Simulated Annealing
ax2.scatter(inst.coords[0, 0], inst.coords[0, 1], c="black", s=140, marker="s", zorder=5, label="Depot")
ax2.scatter(inst.coords[1:, 0], inst.coords[1:, 1], c="gray", s=35, zorder=4, label="Customer")
for r_i, route in enumerate(sa_res.best_routes):
    col = cmap(r_i % 10)
    path = [0, *route, 0]
    pts = inst.coords[path]
    ax2.plot(pts[:, 0], pts[:, 1], color=col, linewidth=2, alpha=0.85)
ax2.set_title(f"Simulated Annealing\nCost: {sa_res.best_cost} (Gap: {sa_gap:.1f}%) | Routes: {len(sa_res.best_routes)}", fontsize=12, fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.4)
ax2.legend(loc="upper left")

fig.suptitle(f"Route Comparison on {inst_name}", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 7. Comparative Performance Across Small, Medium, and Large Tiers
We load the full benchmark results across all 16 CVRPLIB instances and evaluate performance by tier.

In [ ]:
results_file = ROOT / "results" / "heuristic_results.csv"
df = pd.read_csv(results_file)

# Display summary columns
summary_cols = [
    "instance", "tier", "customers", "vehicles", "bks_cost",
    "cw_gap_pct", "cw_time_s", "ls_gap_pct", "ls_time_s",
    "sa_gap_pct", "sa_time_s", "sa_feasible"
]
display(df[summary_cols])

print("\n=== Average Optimality Gap (%) by Tier ===")
print(df.groupby("tier")[["cw_gap_pct", "ls_gap_pct", "sa_gap_pct"]].mean().round(2))

print("\n=== Average Runtime (seconds) by Tier ===")
print(df.groupby("tier")[["cw_time_s", "ls_time_s", "sa_time_s"]].mean().round(4))

## 8. Key Findings & Discussion for Report
1. **Solution Quality:** Across all 16 instances, Simulated Annealing maintains an average gap of **~7% relative to the proven global optimum**.
2. **Computational Scalability:** While Exact MILP fails to scale beyond ~40 customers within a reasonable time limit, the heuristic pipeline solves 255-customer instances (`X-n256-k16`) in **under 0.4 seconds**.
3. **Feasibility:** 100% of generated solutions are verified feasible by `src/checker.py`, adhering strictly to depot return, single-visit, vehicle capacity, and fleet limits.
4. **Algorithmic Progression:** Clarke-Wright provides a rapid initial topology, Local Search strips out obvious route inversions and unneeded vehicle trips, and Simulated Annealing successfully avoids getting trapped in suboptimal local minima.